# MedVQA trên VQA-RAD — notebook tất-cả-trong-một (Colab GPU)

Bài toán: cho ảnh y khoa + câu hỏi tiếng Anh, dự đoán đáp án (classification trên
answer vocab gồm 429 class build từ train). Trục thí nghiệm chính là **fusion**:
`concat` / `hadamard` / `cross_attention`.

> Vì mọi thứ chạy trong process (không subprocess, không `%cd`), nếu Colab mất kết
> nối giữa chừng chỉ cần **Run all** lại — không có lỗi `No module named 'midterm'`.

In [1]:
# torch, torchvision, numpy, PIL có sẵn trên Colab. matplotlib/pandas đã bị gỡ
# khỏi một số runtime Colab mới nên cài lại cho chắc (no-op nếu đã có).
!pip install -q -U "transformers>=4.45" "datasets>=3.0"
!pip install -q matplotlib pandas


## 1. Cấu hình & tiện ích

In [2]:
import gc
import json
import os
import random
import re
from dataclasses import asdict, dataclass

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset


def set_seed(seed=42):
    "Cố định mọi nguồn ngẫu nhiên: 3 thí nghiệm chỉ khác nhau ở fusion."
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class Config:
    # Ảnh / câu hỏi / split
    image_size: int = 224
    max_question_len: int = 32
    val_fraction: float = 0.10
    augment: bool = True
    # Model
    d_model: int = 768
    fusion: str = "concat"            # concat | hadamard | cross_attention
    text_model_name: str = "bert-base-uncased"
    text_pool: str = "mean"           # mean | cls
    unfreeze_last_block: bool = False
    num_heads: int = 8
    hidden_dim: int = 1024
    dropout: float = 0.5
    # Training
    batch_size: int = 64
    lr: float = 1e-3
    lr_backbone: float = 1e-5
    weight_decay: float = 1e-2
    max_epochs: int = 30
    patience: int = 5
    seed: int = 42
    num_workers: int = 2
    # IO (đường dẫn tương đối /content trên Colab)
    run_name: str = ""
    output_dir: str = "outputs"
    checkpoint_dir: str = "checkpoints"

    def __post_init__(self):
        if not self.run_name:
            self.run_name = self.fusion
        assert self.fusion in ("concat", "hadamard", "cross_attention")
        assert self.text_pool in ("mean", "cls")

    def to_dict(self):
        return asdict(self)


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


DEVICE = pick_device()
print("Device:", DEVICE)

Device: cuda


## 2. Dữ liệu: tải VQA-RAD, build vocab, dataset

- Vocab build từ **train** sau chuẩn hóa đáp án — mỗi đáp án duy nhất = 1 class,
  không có `<unk>` (đáp án test ngoài vocab → luôn tính sai, in độ phủ minh bạch).
- Val tách 10% từ train **theo QA pair** (seed 42): VQA-RAD chỉ có 313 ảnh duy nhất
  cho 1.793 câu hỏi và 202/203 ảnh test cũng nằm trong train → split chính thức
  chia theo câu hỏi, nên val theo QA pair khớp đúng điều kiện test.
- **Không** horizontal flip (ảnh y khoa có tính trái/phải).

In [3]:
from datasets import load_dataset
from torchvision import transforms
from transformers import AutoTokenizer

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def normalize_answer(answer):
    "lowercase, bỏ khoảng trắng/dấu câu thừa, gộp khoảng trắng — 'Yes.' -> 'yes'."
    s = answer.strip().lower().strip(".,;:!? ")
    return re.sub(r"\s+", " ", s)


def build_vocab(train_split):
    "Mỗi đáp án duy nhất (sau chuẩn hóa) = 1 class. sorted() để ổn định."
    answers = sorted({normalize_answer(a) for a in train_split["answer"]})
    return {a: i for i, a in enumerate(answers)}


def build_transforms(cfg, train):
    if train and cfg.augment:
        resize = transforms.RandomResizedCrop(cfg.image_size, scale=(0.9, 1.0),
                                              ratio=(1.0, 1.0))
    else:
        resize = transforms.Resize((cfg.image_size, cfg.image_size))
    return transforms.Compose([resize, transforms.ToTensor(),
                               transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])


class VQARadDataset(Dataset):
    def __init__(self, hf_split, tokenizer, vocab, cfg, train):
        self.ds, self.tokenizer, self.vocab, self.cfg = hf_split, tokenizer, vocab, cfg
        self.transform = build_transforms(cfg, train)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        row = self.ds[idx]
        image = self.transform(row["image"].convert("RGB"))
        tok = self.tokenizer(row["question"], padding="max_length", truncation=True,
                             max_length=self.cfg.max_question_len, return_tensors="pt")
        answer = normalize_answer(row["answer"])
        label = self.vocab.get(answer, -1)  # ngoài vocab -> -1: luôn tính sai
        return {"image": image, "input_ids": tok["input_ids"].squeeze(0),
                "attention_mask": tok["attention_mask"].squeeze(0),
                "label": torch.tensor(label, dtype=torch.long),
                "question": row["question"], "answer": answer}


# --- Tải dataset (HF tự cache, chạy lại không tải lại) ---
raw = load_dataset("flaviagiammarino/vqa-rad")
train_full, test_hf = raw["train"], raw["test"]

vocab = build_vocab(train_full)
test_answers = [normalize_answer(a) for a in test_hf["answer"]]
covered = sum(a in vocab for a in test_answers)
print(f"Vocab: {len(vocab)} class | độ phủ test: {covered}/{len(test_answers)}"
      f" = {100 * covered / len(test_answers):.1f}% (trần accuracy khả dĩ)")

split = train_full.train_test_split(test_size=0.10, seed=42)
train_hf, val_hf = split["train"], split["test"]
print(f"Splits: train={len(train_hf)} | val={len(val_hf)} | test={len(test_hf)}")

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

C:\jupyterhub_windows\jupyterenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vocab: 429 class | độ phủ test: 334/451 = 74.1% (trần accuracy khả dĩ)
Splits: train=1613 | val=180 | test=451


## 3. Model: encoder (freeze) + fusion + head

ResNet-50 và BERT đều freeze (BatchNorm/dropout ghim ở eval để running stats không
trôi theo ảnh y khoa, kết quả tái lập). Ba fusion cùng interface
`(v_img, img_map, v_txt) -> (fused, attn|None)` nên đổi fusion chỉ đổi config.

In [4]:
from torchvision.models import ResNet50_Weights, resnet50
from transformers import AutoModel


class ImageEncoder(nn.Module):
    "ResNet-50 ImageNet, freeze. Xuất vector toàn cục (B,d) + 49 vùng (B,49,d)."

    def __init__(self, d_model=768, unfreeze_last_block=False):
        super().__init__()
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # -> (B,2048,7,7)
        for p in self.backbone.parameters():
            p.requires_grad = False
        if unfreeze_last_block:
            for p in self.backbone[-1].parameters():
                p.requires_grad = True
        self.proj_global = nn.Linear(2048, d_model)
        self.proj_regions = nn.Linear(2048, d_model)

    def train(self, mode=True):
        # BatchNorm LUÔN eval: requires_grad=False chỉ chặn weight, không chặn
        # running stats — nếu để train mode chúng vẫn trôi theo ảnh y khoa.
        super().train(mode)
        self.backbone.eval()
        return self

    def forward(self, images):
        fmap = self.backbone(images)                # (B,2048,7,7)
        v_global = fmap.mean(dim=(2, 3))            # (B,2048)
        regions = fmap.flatten(2).transpose(1, 2)   # (B,49,2048)
        return self.proj_global(v_global), self.proj_regions(regions)


class TextEncoder(nn.Module):
    "BERT freeze. mean-pooling (mặc định, tốt hơn [CLS] khi không fine-tune) hoặc cls."

    def __init__(self, model_name="bert-base-uncased", pool="mean"):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.pool = pool
        for p in self.bert.parameters():
            p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.bert.eval()  # tắt dropout: biểu diễn câu ổn định giữa các batch
        return self

    def forward(self, input_ids, attention_mask):
        hidden = self.bert(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state
        if self.pool == "cls":
            return hidden[:, 0]
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype)
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


class ConcatFusion(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.fc = nn.Linear(2 * d_model, d_model)
        self.relu = nn.ReLU()

    def forward(self, v_img, img_map, v_txt):
        return self.relu(self.fc(torch.cat([v_img, v_txt], dim=-1))), None


class HadamardFusion(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.proj_img = nn.Linear(d_model, d_model)
        self.proj_txt = nn.Linear(d_model, d_model)

    def forward(self, v_img, img_map, v_txt):
        return self.proj_img(v_img) * self.proj_txt(v_txt), None


class CrossAttentionFusion(nn.Module):
    "Câu hỏi (query) nhìn vào 49 vùng ảnh (key/value) + residual + LayerNorm."

    def __init__(self, d_model=768, num_heads=8):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, v_img, img_map, v_txt):
        attended, weights = self.attn(v_txt.unsqueeze(1), img_map, img_map)
        fused = self.norm(v_txt + attended.squeeze(1))
        return fused, weights.squeeze(1)  # attn (B,49) để vẽ heatmap


def build_fusion(cfg):
    if cfg.fusion == "concat":
        return ConcatFusion(cfg.d_model)
    if cfg.fusion == "hadamard":
        return HadamardFusion(cfg.d_model)
    return CrossAttentionFusion(cfg.d_model, cfg.num_heads)


class VQAModel(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__()
        self.image_encoder = ImageEncoder(cfg.d_model, cfg.unfreeze_last_block)
        self.text_encoder = TextEncoder(cfg.text_model_name, cfg.text_pool)
        self.fusion = build_fusion(cfg)
        self.head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.hidden_dim), nn.ReLU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.hidden_dim, num_classes))

    def forward(self, images, input_ids, attention_mask):
        v_img, img_map = self.image_encoder(images)
        v_txt = self.text_encoder(input_ids, attention_mask)
        fused, attn = self.fusion(v_img, img_map, v_txt)
        return self.head(fused), attn

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

## 4. Vòng train + đánh giá

Hai param group: phần tự xây (fusion + projection + head) học `lr=1e-3`; layer4
ResNet (nếu unfreeze) học `lr=1e-5`. AdamW + cosine decay, early stopping theo val
overall accuracy (patience 5), lưu checkpoint tốt nhất rồi nạp lại trước khi trả về.

In [5]:
import matplotlib.pyplot as plt


def plot_curves(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ep = range(1, len(history["train_loss"]) + 1)
    ax1.plot(ep, history["train_loss"], label="train")
    ax1.plot(ep, history["val_loss"], label="val")
    ax1.set_title(f"{title} — loss"); ax1.set_xlabel("epoch"); ax1.legend()
    ax2.plot(ep, history["val_acc"]); ax2.set_title(f"{title} — val accuracy")
    ax2.set_xlabel("epoch")
    plt.tight_layout(); plt.show()


@torch.no_grad()
def run_validation(model, loader, device):
    model.eval()
    crit = nn.CrossEntropyLoss()
    correct = total = 0
    loss_sum = 0.0
    for b in loader:
        labels = b["label"].to(device)
        logits, _ = model(b["image"].to(device), b["input_ids"].to(device),
                          b["attention_mask"].to(device))
        loss_sum += crit(logits, labels).item() * labels.size(0)
        correct += (logits.argmax(-1) == labels).sum().item()
        total += labels.size(0)
    return correct / total, loss_sum / total


def train(cfg, train_hf, val_hf, vocab, tokenizer, device):
    set_seed(cfg.seed)
    print(f"=== {cfg.run_name} | fusion={cfg.fusion} | text_pool={cfg.text_pool} ===")
    train_loader = DataLoader(VQARadDataset(train_hf, tokenizer, vocab, cfg, True),
                              batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers)
    val_loader = DataLoader(VQARadDataset(val_hf, tokenizer, vocab, cfg, False),
                            batch_size=cfg.batch_size, shuffle=False,
                            num_workers=cfg.num_workers)

    model = VQAModel(cfg, num_classes=len(vocab)).to(device)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.trainable_parameters())
    print(f"Tham số: {total / 1e6:.0f}M tổng | {trainable / 1e6:.1f}M trainable")

    backbone = [p for p in model.image_encoder.backbone.parameters() if p.requires_grad]
    bb_ids = {id(p) for p in backbone}
    new_params = [p for p in model.trainable_parameters() if id(p) not in bb_ids]
    groups = [{"params": new_params, "lr": cfg.lr}]
    if backbone:
        groups.append({"params": backbone, "lr": cfg.lr_backbone})
    opt = torch.optim.AdamW(groups, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.max_epochs)
    crit = nn.CrossEntropyLoss()

    os.makedirs(os.path.join(cfg.output_dir, cfg.run_name), exist_ok=True)
    os.makedirs(cfg.checkpoint_dir, exist_ok=True)
    ckpt_path = os.path.join(cfg.checkpoint_dir, f"{cfg.run_name}.pt")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_acc = 0.0
    no_improve = 0
    for epoch in range(1, cfg.max_epochs + 1):
        model.train()  # encoder tự ghim eval bên trong
        loss_sum = seen = 0
        for b in train_loader:
            labels = b["label"].to(device)
            opt.zero_grad()
            logits, _ = model(b["image"].to(device), b["input_ids"].to(device),
                              b["attention_mask"].to(device))
            loss = crit(logits, labels)
            loss.backward()
            opt.step()
            loss_sum += loss.item() * labels.size(0)
            seen += labels.size(0)
        sched.step()
        val_acc, val_loss = run_validation(model, val_loader, device)
        history["train_loss"].append(loss_sum / seen)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch:02d} | train {loss_sum / seen:.4f} | val loss {val_loss:.4f}"
              f" | val acc {val_acc:.4f}")
        if val_acc > best_acc:
            best_acc, no_improve = val_acc, 0
            torch.save({"model_state": model.state_dict(), "config": cfg.to_dict(),
                        "num_classes": len(vocab)}, ckpt_path)
        else:
            no_improve += 1
            if no_improve >= cfg.patience:
                print(f"Early stopping @ epoch {epoch} (patience {cfg.patience})")
                break

    json.dump(history,
              open(os.path.join(cfg.output_dir, cfg.run_name, "history.json"), "w"))
    model.load_state_dict(torch.load(ckpt_path)["model_state"])  # nạp lại best
    print(f"Best val acc: {best_acc:.4f}")
    return model, history


@torch.no_grad()
def evaluate(model, cfg, test_hf, vocab, tokenizer, device):
    idx_to_answer = {i: a for a, i in vocab.items()}
    loader = DataLoader(VQARadDataset(test_hf, tokenizer, vocab, cfg, False),
                        batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    model.eval()
    rows = []
    for b in loader:
        logits, _ = model(b["image"].to(device), b["input_ids"].to(device),
                          b["attention_mask"].to(device))
        preds = logits.argmax(-1).cpu()
        for q, a, lab, pr in zip(b["question"], b["answer"], b["label"], preds):
            rows.append({"question": q, "answer": a, "pred": idx_to_answer[pr.item()],
                         "correct": pr.item() == lab.item(), "closed": a in ("yes", "no")})

    def acc(subset):
        return sum(r["correct"] for r in subset) / len(subset) if subset else 0.0

    closed = [r for r in rows if r["closed"]]
    opened = [r for r in rows if not r["closed"]]
    metrics = {"overall": acc(rows), "closed": acc(closed), "open": acc(opened),
               "n": len(rows), "n_closed": len(closed), "n_open": len(opened)}
    print(f"[{cfg.fusion}] overall {metrics['overall']:.4f} |"
          f" closed {metrics['closed']:.4f} (n={metrics['n_closed']}) |"
          f" open {metrics['open']:.4f} (n={metrics['n_open']})")
    json.dump({"metrics": metrics, "rows": rows},
              open(os.path.join(cfg.output_dir, cfg.run_name, "test_results.json"), "w"),
              ensure_ascii=False, indent=2)
    return metrics, rows


def load_model(ckpt_path, device):
    "Dựng lại model từ config lưu trong checkpoint (cho demo)."
    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = Config(**ckpt["config"])
    model = VQAModel(cfg, num_classes=ckpt["num_classes"])
    model.load_state_dict(ckpt["model_state"])
    return model.to(device).eval(), cfg

## 5. Chạy 3 thí nghiệm fusion

Mỗi fusion: train → đánh giá test → vẽ learning curve. Giải phóng model sau mỗi
vòng để tiết kiệm VRAM. Tổng thời gian ~30–60 phút trên T4.

In [ ]:
results = {}
for fusion in ["concat", "hadamard", "cross_attention"]:
    cfg = Config(fusion=fusion)
    model, history = train(cfg, train_hf, val_hf, vocab, tokenizer, DEVICE)
    metrics, _ = evaluate(model, cfg, test_hf, vocab, tokenizer, DEVICE)
    results[fusion] = metrics
    plot_curves(history, fusion)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

=== concat | fusion=concat | text_pool=mean ===


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10352.32it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tham số: 139M tổng | 5.6M trainable


## 6. Bảng kết quả (overall / closed / open)

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"fusion": f, "overall": m["overall"], "closed (yes/no)": m["closed"], "open": m["open"]}
    for f, m in results.items()
]).set_index("fusion").round(4)
df

## 7. Demo + attention overlay (cross_attention)

Lấy 1 ảnh test làm ví dụ: in top-5 đáp án kèm xác suất và vẽ heatmap attention (49
vùng 7×7 phóng to) chồng lên ảnh — minh họa model "nhìn" vào đâu khi trả lời.

In [ ]:
model, cfg = load_model("checkpoints/cross_attention.pt", DEVICE)
sample = test_hf[0]
image, question = sample["image"].convert("RGB"), sample["question"]
print("Q:", question, "| GT:", normalize_answer(sample["answer"]))

pixel = build_transforms(cfg, train=False)(image).unsqueeze(0).to(DEVICE)
tok = tokenizer(question, padding="max_length", truncation=True,
                max_length=cfg.max_question_len, return_tensors="pt")
with torch.no_grad():
    logits, attn = model(pixel, tok["input_ids"].to(DEVICE),
                         tok["attention_mask"].to(DEVICE))

idx_to_answer = {i: a for a, i in vocab.items()}
top = logits.softmax(-1).squeeze(0).topk(5)
print("Top-5:")
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"  {idx_to_answer[i]:<28s} {p:.3f}")

heat = np.kron(attn.squeeze(0).reshape(7, 7).cpu().numpy(), np.ones((32, 32)))
plt.figure(figsize=(5, 5))
plt.imshow(image.resize((224, 224)))
plt.imshow(heat, cmap="jet", alpha=0.4)
plt.title(question, fontsize=9)
plt.axis("off")
plt.show()

## 8. Tải kết quả về máy (tùy chọn)

In [ ]:
!zip -r -q results.zip outputs checkpoints
from google.colab import files

files.download("results.zip")